In [1]:
import os
import json

In [2]:

# Target shot cases and canonical models
SHOT_CASES = [1, 5, 10, 25]
MODELS = [
    "AVES-bio",
    "BirdAVES-biox-base",
    "DistilHuBERT",
    "HuBERT",
    "MusicHuBERT",
    "WavLM"
]

# (Metric Display Name, folder_name, json_filename, json_key)
METRICS = [
    ("LogME", "logME", "all_models_logme.json", "overall_logme"),
    ("H-score", "H_score", "all_models_hscore.json", "overall_hscore"),
    ("LFC", "LFC_score", "all_models_lfc.json", "overall_lfc"),
    ("LGC (Linear)", "LGC_linear", "all_models_lgc.json", "overall_lgc")
]

# Determine repository base path
base_prefix = ".." if os.path.exists("../model_embeddings") else "."
output_tex_path = os.path.join(base_prefix, "tables_transferability.tex")

def locate_metric_file(shot: int, folder: str, filename: str):
    candidates = [
        os.path.join(base_prefix, f"model_scores_{shot}_shot", folder, filename),
        os.path.join(base_prefix, "model_scores", folder, filename)
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

latex_output = []

for shot in SHOT_CASES:
    # 1. Load metric data for this shot
    metric_data = {}
    for label, folder, filename, key in METRICS:
        filepath = locate_metric_file(shot, folder, filename)
        if filepath and os.path.exists(filepath):
            with open(filepath, "r") as f:
                metric_data[label] = (json.load(f), key)
        else:
            metric_data[label] = ({}, key)

    # 2. Build LaTeX Table
    table_str = [
        r"\begin{table}[htbp]",
        r"\centering",
        f"\\caption{{Transferability metric scores across backbone models under the {shot}-shot regime.}}",
        f"\\label{{tab:transferability_{shot}_shot}}",
        r"\begin{tabular}{lcccc}",
        r"\hline",
        r"\textbf{Backbone Model} & \textbf{LogME} & \textbf{H-score} & \textbf{LFC} & \textbf{LGC (Linear)} \\ \hline"
    ]

    for model in MODELS:
        row = [f"\\texttt{{{model}}}"]
        for label, _, _, _ in METRICS:
            data_dict, key = metric_data[label]
            if model in data_dict and key in data_dict[model]:
                val = data_dict[model][key]
                row.append(f"{val:.4f}")
            else:
                row.append("--")
        table_str.append(" & ".join(row) + r" \\")

    table_str.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table}",
        "\n"
    ])

    latex_output.append("\n".join(table_str))

# 3. Write all tables to file
with open(output_tex_path, "w") as f:
    f.write("\n".join(latex_output))

print(f"Generated 4 LaTeX tables successfully in: {output_tex_path}")

Generated 4 LaTeX tables successfully in: .\tables_transferability.tex


### Test accuracy and loss

In [6]:
import os
import json

# Target shot cases and canonical model list
SHOT_CASES = [1, 5, 10, 25]
MODELS = [
    "AVES-bio",
    "BirdAVES-biox-base",
    "DistilHuBERT",
    "HuBERT",
    "MusicHuBERT",
    "WavLM"
]

# Path resolution across repo root or subdirectories
base_prefix = ".." if os.path.exists("../model_embeddings_eval") else "."
reg_eval_dir = os.path.join(base_prefix, "model_eval_results")
epoch_eval_dir = os.path.join(base_prefix, "model_eval_results_epoch_tuned")
output_tex_path = os.path.join(base_prefix, "tables_eval_performance.tex")

latex_tables = []

for shot in SHOT_CASES:
    reg_file = os.path.join(reg_eval_dir, f"linear_eval_{shot}_shot.json")
    epoch_file = os.path.join(epoch_eval_dir, f"linear_eval_{shot}_shot.json")

    reg_data = {}
    if os.path.exists(reg_file):
        with open(reg_file, "r") as f:
            reg_data = json.load(f)

    epoch_data = {}
    if os.path.exists(epoch_file):
        with open(epoch_file, "r") as f:
            epoch_data = json.load(f)

    # Build the LaTeX table block
    table_lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        f"\\caption{{Evaluation performance on unseen test data under the {shot}-shot regime across training strategies.}}",
        f"\\label{{tab:eval_performance_{shot}_shot}}",
        r"\begin{tabular}{lcccc}",
        r"\hline",
        r"\textbf{Backbone Model} & \multicolumn{2}{c}{\textbf{Regularization-Tuned}} & \multicolumn{2}{c}{\textbf{Optimal Epoch-Tuned}} \\",
        r"\cline{2-5}",
        r"& \textbf{Accuracy (\%)} & \textbf{Test Loss} & \textbf{Accuracy (\%)} & \textbf{Test Loss} \\",
        r"\hline"
    ]

    for model in MODELS:
        # Regularization-tuned values
        if model in reg_data:
            r_acc = f"{reg_data[model]['overall_accuracy'] * 100:.2f}"
            r_loss = f"{reg_data[model]['test_loss']:.4f}"
        else:
            r_acc, r_loss = "--", "--"

        # Epoch-tuned values
        if model in epoch_data:
            e_acc = f"{epoch_data[model]['overall_accuracy'] * 100:.2f}"
            e_loss = f"{epoch_data[model]['test_loss']:.4f}"
        else:
            e_acc, e_loss = "--", "--"

        table_lines.append(
            f"\\texttt{{{model}}} & {r_acc} & {r_loss} & {e_acc} & {e_loss} \\\\"
        )

    table_lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table}",
        "\n"
    ])

    latex_tables.append("\n".join(table_lines))

# Write all tables to file
with open(output_tex_path, "w") as f:
    f.write("\n".join(latex_tables))

print(f"Successfully generated all evaluation tables in: {output_tex_path}")

Successfully generated all evaluation tables in: .\tables_eval_performance.tex


### Rank comparison table generation

In [7]:
import os
import json
import numpy as np
import pandas as pd
from scipy.stats import rankdata

# Target shot cases and canonical models
SHOT_CASES = [1, 5, 10, 25]
MODELS = [
    "AVES-bio",
    "BirdAVES-biox-base",
    "DistilHuBERT",
    "HuBERT",
    "MusicHuBERT",
    "WavLM"
]

# Transferability metric specifications: (Display Name, folder_name, json_filename, json_key)
METRICS = [
    ("LogME", "logME", "all_models_logme.json", "overall_logme"),
    ("H-score", "H_score", "all_models_hscore.json", "overall_hscore"),
    ("LFC", "LFC_score", "all_models_lfc.json", "overall_lfc"),
    ("LGC", "LGC_linear", "all_models_lgc.json", "overall_lgc")
]

# Path resolution across repo root or subdirectories
base_prefix = ".." if os.path.exists("../model_embeddings") else "."
eval_dir = os.path.join(base_prefix, "model_eval_results")
output_tex_path = os.path.join(base_prefix, "tables_rank_comparison.tex")

def locate_metric_file(shot: int, folder: str, filename: str):
    """
    Locates metric JSON files checking shot-specific and root score paths.
    """
    candidates = [
        os.path.join(base_prefix, f"model_scores_{shot}_shot", folder, filename),
        os.path.join(base_prefix, "model_scores", folder, filename)
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

def compute_ranks_descending(values: list):
    """
    Ranks values in descending order (highest score/accuracy = Rank 1).
    Handles NaNs by assigning '--'.
    """
    arr = np.array(values, dtype=float)
    if np.all(np.isnan(arr)):
        return ["--"] * len(values)
    
    # rankdata assigns 1 to lowest; negate so highest value gets 1
    # method='min' handles potential ties consistently
    ranks = rankdata(-arr, method="min")
    formatted_ranks = []
    for orig_val, r in zip(arr, ranks):
        if np.isnan(orig_val):
            formatted_ranks.append("--")
        else:
            formatted_ranks.append(str(int(r)))
    return formatted_ranks

latex_tables = []

for shot in SHOT_CASES:
    eval_file = os.path.join(eval_dir, f"linear_eval_{shot}_shot.json")
    
    # 1. Load evaluation test accuracies (explicit regularisation)
    test_accs = {model: np.nan for model in MODELS}
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            eval_data = json.load(f)
        for model in MODELS:
            if model in eval_data and "overall_accuracy" in eval_data[model]:
                test_accs[model] = eval_data[model]["overall_accuracy"]
    else:
        print(f"Notice: Evaluation file not found for {shot}-shot ({eval_file}).")

    # 2. Load transferability scores
    metric_values = {label: {model: np.nan for model in MODELS} for label, _, _, _ in METRICS}
    for label, folder, filename, key in METRICS:
        filepath = locate_metric_file(shot, folder, filename)
        if filepath and os.path.exists(filepath):
            with open(filepath, "r") as f:
                data = json.load(f)
            for model in MODELS:
                if model in data and key in data[model]:
                    metric_values[label][model] = data[model][key]

    # 3. Compute descending ranks across models
    acc_list = [test_accs[m] for m in MODELS]
    acc_ranks = compute_ranks_descending(acc_list)

    metric_ranks = {}
    for label, _, _, _ in METRICS:
        vals = [metric_values[label][m] for m in MODELS]
        metric_ranks[label] = compute_ranks_descending(vals)

    # 4. Construct LaTeX table block
    table_lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        f"\\caption{{Model ranking comparison between analytical transferability metrics and held-out test accuracy under the {shot}-shot regime (regularized linear probe).}}",
        f"\\label{{tab:rank_comparison_{shot}_shot}}",
        r"\begin{tabular}{lccccc}",
        r"\hline",
        r"\textbf{Backbone Model} & \multicolumn{4}{c}{\textbf{Transferability Metric Ranks}} & \textbf{Test Acc. Rank} \\",
        r"\cline{2-5}",
        r"& \textbf{LogME} & \textbf{H-score} & \textbf{LFC} & \textbf{LGC} & \\",
        r"\hline"
    ]

    for idx, model in enumerate(MODELS):
        row = [f"\\texttt{{{model}}}"]
        for label, _, _, _ in METRICS:
            row.append(metric_ranks[label][idx])
        row.append(acc_ranks[idx])
        table_lines.append(" & ".join(row) + r" \\")

    table_lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table}",
        "\n"
    ])

    latex_tables.append("\n".join(table_lines))

# 5. Export generated tables to .tex file
with open(output_tex_path, "w") as f:
    f.write("\n".join(latex_tables))

print(f"Generated rank comparison tables successfully in: {output_tex_path}")

Generated rank comparison tables successfully in: .\tables_rank_comparison.tex


### Rank correlation metrics table

In [9]:
import os
import json

# Target shot cases and transferability score estimators (columns)
SHOT_CASES = [1, 5, 10, 25]
TRANSFERABILITY_METRICS = ["LogME", "H-score", "LFC", "LGC"]

# Correlation statistical metrics (rows)
# Tuple format: (Display Name, JSON key)
STAT_METRICS = [
    (r"Spearman's $\rho$", "spearman_rho"),
    (r"$p$-value ($\rho$)", "spearman_pvalue"),
    (r"Kendall's $\tau$", "kendall_tau"),
    (r"$p$-value ($\tau$)", "kendall_pvalue"),
    (r"Pearson's $r$", "pearson_r"),
    (r"$p$-value ($r$)", "pearson_pvalue")
]

# Path resolution across repository root or subdirectories
base_prefix = ".." if os.path.exists("../model_embeddings") else "."
corr_json_path = os.path.join(base_prefix, "correlation_results", "rank_correlations_regularized.json")
output_tex_path = os.path.join(base_prefix, "tables_correlation_metrics.tex")

# 1. Load precomputed correlation results
if not os.path.exists(corr_json_path):
    raise FileNotFoundError(f"Correlation JSON file not found at: {corr_json_path}")

with open(corr_json_path, "r") as f:
    corr_data = json.load(f)

# Structure lookup table: lookup[shot][metric_name] = item
lookup = {}
for entry in corr_data.get("correlations", []):
    s = entry["shots"]
    m = entry["metric"]
    if s not in lookup:
        lookup[s] = {}
    lookup[s][m] = entry

latex_tables = []

# 2. Build LaTeX tables for each shot case
for shot in SHOT_CASES:
    if shot not in lookup:
        print(f"Skipping {shot}-shot: No correlation records found in JSON.")
        continue

    shot_records = lookup[shot]

    table_lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        f"\\caption{{Correlation metrics between transferability scores and regularized test accuracy under the {shot}-shot regime.}}",
        f"\\label{{tab:correlation_metrics_{shot}_shot}}",
        r"\begin{tabular}{lcccc}",
        r"\hline",
        r"\textbf{Correlation Metric} & \textbf{LogME} & \textbf{H-score} & \textbf{LFC} & \textbf{LGC} \\ \hline"
    ]

    for label, json_key in STAT_METRICS:
        row = [label]
        for t_metric in TRANSFERABILITY_METRICS:
            if t_metric in shot_records and json_key in shot_records[t_metric]:
                val = shot_records[t_metric][json_key]
                # Format p-values using scientific notation if extremely small
                if "pvalue" in json_key and abs(val) < 0.001:
                    formatted_val = f"{val:.2e}"
                else:
                    formatted_val = f"{val:.4f}"
                row.append(formatted_val)
            else:
                row.append("--")
        table_lines.append(" & ".join(row) + r" \\")

    table_lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table}",
        "\n"
    ])

    latex_tables.append("\n".join(table_lines))

# 3. Save all tables to file
with open(output_tex_path, "w") as f:
    f.write("\n".join(latex_tables))

print(f"Successfully generated correlation metric tables in: {output_tex_path}")

Successfully generated correlation metric tables in: .\tables_correlation_metrics.tex


### Lambda used table

In [5]:
import json
import os


def generate_latex_lambda_table(
    shot_sizes=[5, 10, 25],
    base_dir=".",
    caption="Optimal weight decay parameter ($\\lambda^*$) selected via cross-validation across few-shot regimes.",
    label="tab:optimal_lambdas",
):
    """Parses training_summary.json across shot sizes and outputs a clean LaTeX table."""
    data = {}

    for k in shot_sizes:
        # Check standard relative paths
        summary_path = os.path.join(
            base_dir, f"trained_models_{k}_shot", "Linear", "training_summary.json"
        )
        if not os.path.exists(summary_path):
            summary_path = os.path.join(
                base_dir,
                f"../trained_models_{k}_shot",
                "Linear",
                "training_summary.json",
            )

        if not os.path.exists(summary_path):
            print(f"Warning: File not found for K={k}: {summary_path}")
            continue

        with open(summary_path, "r") as f:
            summary = json.load(f)

        for model_name, details in summary.items():
            if model_name not in data:
                data[model_name] = {}
            data[model_name][k] = details.get("optimal_lambda", None)

    if not data:
        print("No summary data found.")
        return

    # LaTeX column alignment string (left-aligned model name, centered columns for shots)
    col_align = "l" + "c" * len(shot_sizes)

    # Format lambda cleanly for LaTeX math mode
    def fmt_lambda(val):
        if val is None:
            return "--"
        try:
            val = float(val)
        except (ValueError, TypeError):
            return str(val)

        if val < 0.01:
            # Parse scientific notation exponent as integer
            exp = int(f"{val:e}".split("e")[1])
            return f"$10^{{{exp}}}$"
        return f"${val}$"

    # Header row
    headers = [f"\\textbf{{$K={k}$}}" for k in shot_sizes]
    header_row = (
        "\\textbf{Backbone Model} & " + " & ".join(headers) + " \\\\ \\hline"
    )

    # Build LaTeX table
    latex_code = [
        "\\begin{table}[H]",
        "\\centering",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        f"\\begin{{tabular}}{{{col_align}}}",
        "\\hline",
        header_row,
    ]

    for model_name in sorted(data.keys()):
        # Escape underscores for LaTeX compatibility
        clean_model_name = (
            f"\\texttt{{{model_name.replace('_', r'\\_')}}}"
        )
        row_vals = [fmt_lambda(data[model_name].get(k)) for k in shot_sizes]
        latex_code.append(
            f"{clean_model_name} & " + " & ".join(row_vals) + " \\\\"
        )

    latex_code.extend(["\\hline", "\\end{tabular}", "\\end{table}"])

    latex_table_str = "\n".join(latex_code)
    print(latex_table_str)
    return latex_table_str


if __name__ == "__main__":
    generate_latex_lambda_table()

\begin{table}[H]
\centering
\caption{Optimal weight decay parameter ($\lambda^*$) selected via cross-validation across few-shot regimes.}
\label{tab:optimal_lambdas}
\begin{tabular}{lccc}
\hline
\textbf{Backbone Model} & \textbf{$K=5$} & \textbf{$K=10$} & \textbf{$K=25$} \\ \hline
\texttt{AVES-bio} & $1.0$ & $10^{-4}$ & $10^{-4}$ \\
\texttt{BirdAVES-biox-base} & $0.1$ & $10^{-4}$ & $10^{-4}$ \\
\texttt{DistilHuBERT} & $1.0$ & $10^{-4}$ & $10^{-4}$ \\
\texttt{HuBERT} & $10.0$ & $10^{-4}$ & $10^{-4}$ \\
\texttt{MusicHuBERT} & $1.0$ & $0.1$ & $10^{-4}$ \\
\texttt{WavLM} & $1.0$ & $0.01$ & $10^{-4}$ \\
\hline
\end{tabular}
\end{table}


### Tabe for number of epochs

In [1]:
import json
import os


def generate_latex_epoch_table(
    shot_sizes=[5, 10, 25],
    base_dir=".",
    caption="Optimal training epochs ($T^*$) determined via 5-fold cross-validation early stopping across few-shot regimes.",
    label="tab:optimal_epochs",
):
    """Parses training_summary.json from trained_models_epoch_tuned_{K}_shot directories

    and outputs a publication-grade LaTeX table.
    """
    data = {}

    for k in shot_sizes:
        summary_path = os.path.join(
            base_dir,
            f"trained_models_epoch_tuned_{k}_shot",
            "Linear",
            "training_summary.json",
        )
        if not os.path.exists(summary_path):
            summary_path = os.path.join(
                base_dir,
                f"../trained_models_epoch_tuned_{k}_shot",
                "Linear",
                "training_summary.json",
            )

        if not os.path.exists(summary_path):
            print(f"Warning: File not found for K={k}: {summary_path}")
            continue

        with open(summary_path, "r") as f:
            summary = json.load(f)

        for model_name, details in summary.items():
            if model_name not in data:
                data[model_name] = {}
            data[model_name][k] = details.get("optimal_epoch", None)

    if not data:
        print("No summary data found.")
        return ""

    col_align = "l" + "c" * len(shot_sizes)

    def fmt_epoch(val):
        if val is None:
            return "--"
        return f"${int(val)}$"

    headers = [f"\\textbf{{$K={k}$}}" for k in shot_sizes]
    header_row = (
        "\\textbf{Backbone Model} & " + " & ".join(headers) + " \\\\ \\hline"
    )

    latex_code = [
        "\\begin{table}[H]",
        "\\centering",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        f"\\begin{{tabular}}{{{col_align}}}",
        "\\hline",
        header_row,
    ]

    for model_name in sorted(data.keys()):
        clean_model_name = f"\\texttt{{{model_name.replace('_', r'\\_')}}}"
        row_vals = [fmt_epoch(data[model_name].get(k)) for k in shot_sizes]
        latex_code.append(
            f"{clean_model_name} & " + " & ".join(row_vals) + " \\\\"
        )

    latex_code.extend(["\\hline", "\\end{tabular}", "\\end{table}"])

    latex_table_str = "\n".join(latex_code)
    print(latex_table_str)
    return latex_table_str


if __name__ == "__main__":
    generate_latex_epoch_table()

\begin{table}[H]
\centering
\caption{Optimal training epochs ($T^*$) determined via 5-fold cross-validation early stopping across few-shot regimes.}
\label{tab:optimal_epochs}
\begin{tabular}{lccc}
\hline
\textbf{Backbone Model} & \textbf{$K=5$} & \textbf{$K=10$} & \textbf{$K=25$} \\ \hline
\texttt{AVES-bio} & $62$ & $150$ & $150$ \\
\texttt{BirdAVES-biox-base} & $90$ & $150$ & $150$ \\
\texttt{DistilHuBERT} & $57$ & $133$ & $150$ \\
\texttt{HuBERT} & $31$ & $105$ & $150$ \\
\texttt{MusicHuBERT} & $41$ & $95$ & $150$ \\
\texttt{WavLM} & $46$ & $96$ & $150$ \\
\hline
\end{tabular}
\end{table}
